In [1]:
# ==========================================================
# Notebook 00 — Data Setup and Archive Extraction
# ==========================================================
#
# Purpose:
# Prepare the raw Department for Education source files for
# downstream processing by extracting annual ZIP archives
# and standardising filenames into a single processed-data
# directory.
#
# Inputs:
# - Raw ZIP archives stored in the data/raw directory:
#   - 16-17.zip
#   - 17-18.zip
#   - 18-19.zip
#   - 20-21.zip
#   - 21-22.zip
#   - 22-23.zip
#   - 23-24.zip
# - Loose CSV files for the 2019–2020 cycle:
#   - absence_1920.csv
#   - census_1920.csv
#
# Outputs:
# - Extracted and renamed annual source files written to:
#   - data/processed_data/
#
# Role in the pipeline:
# This is the first operational notebook in the pipeline.
# It creates a consistent processed-data folder structure
# used by subsequent ingestion and harmonisation notebooks.
#
# Reproducibility note:
# The file paths in this notebook are currently configured
# for the author’s local machine. These should be updated
# if the project is run in a different environment.
#
# Important:
# This notebook was developed on a local Windows environment.
# Users reproducing the pipeline should replace absolute paths
# with environment-specific paths or relative project paths.
# ==========================================================


import os
import zipfile
import shutil

# ==========================================================
# Configuration
# ==========================================================
#
# This section defines the local source and output directories
# used for file extraction.
#
# - source_folder points to the location of the raw ZIP
#   archives and loose CSV files.
# - output_folder is the destination folder for extracted
#   and renamed files.
#
# If reproducing this pipeline on another machine, update
# these paths before running the notebook.
# ==========================================================

# --- CONFIGURATION ---
source_folder = r'C:\Users\kiero\Documents\msc-dissertation\data\raw'
output_folder = r'C:\Users\kiero\Documents\msc-dissertation\data\processed_data'

# Create the output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# List of your zip files
zip_files = [
    '16-17.zip', '17-18.zip', '18-19.zip',
    '20-21.zip', '21-22.zip', '22-23.zip', '23-24.zip'
]

print(f"Starting extraction from:\n  {source_folder}\nTo:\n  {output_folder}\n")
print("-" * 50)

# ==========================================================
# Part 1 — Extract and Rename Archived Annual Files
# ==========================================================
#
# This step loops through each annual ZIP archive, extracts
# all non-hidden files, and appends a year suffix to each
# filename.
#
# Why the renaming is necessary:
# The Department for Education archive structure varies by
# year, and multiple files may otherwise share similar names.
# Appending the year identifier ensures that files remain
# uniquely identifiable after extraction.
#
# Example:
# - census.csv inside 16-17.zip becomes census_1617.csv
# - absence.csv inside 22-23.zip becomes absence_2223.csv
# ==========================================================

# --- PART 1: EXTRACT AND RENAME ZIP FILES ---
for z_filename in zip_files:
    # 1. Get the year identifier (e.g., "16-17" -> "1617")
    year_id = z_filename.replace('.zip', '').replace('-', '') 
    zip_path = os.path.join(source_folder, z_filename)

    if not os.path.exists(zip_path):
        print(f" Skipped {z_filename} (File not found)")
        continue

    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            for original_path in z.namelist():
                # Skip directories inside the zip
                if original_path.endswith('/'): 
                    continue
                
                # --- THE FIX IS HERE ---
                # We strip the folder path inside the zip to get just the filename
                filename_only = os.path.basename(original_path)
                
                # Skip if it's a hidden file like .DS_Store or empty
                if not filename_only or filename_only.startswith('.'):
                    continue

                # 2. Create the new name (e.g., england_census_1617.csv)
                name_parts = os.path.splitext(filename_only)
                new_filename = f"{name_parts[0]}_{year_id}{name_parts[1]}"

                # 3. Extract to the new clean folder
                source = z.open(original_path)
                target_path = os.path.join(output_folder, new_filename)

                with open(target_path, "wb") as target:
                    shutil.copyfileobj(source, target)

                print(f" Extracted: {new_filename}")

    except zipfile.BadZipFile:
        print(f" Error: {z_filename} appears to be corrupted.")

# ==========================================================
# Part 2 — Copy Non-Archived 2019–2020 Files
# ==========================================================
#
# The 2019–2020 source files are handled separately because,
# in the working dataset used for this project, these files
# were stored as loose CSVs rather than as part of a ZIP archive.
#
# This step copies those files directly into the processed-data
# directory so that all years are available in a common location
# for later ingestion.
# ==========================================================

# --- PART 2: MOVE THE LOOSE 19-20 FILES ---
loose_files = ['absence_1920.csv', 'census_1920.csv']

print("-" * 50)
for file_name in loose_files:
    src = os.path.join(source_folder, file_name)
    dst = os.path.join(output_folder, file_name)
    
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f" Copied: {file_name} (19-20 data)")
    else:
        print(f" Could not find loose file: {file_name}")

print("\nProcessing complete! Check the 'processed_data' folder.")

# ==========================================================
# Outputs Produced
# ==========================================================
#
# After successful execution, this notebook produces a
# populated processed_data directory containing:
#
# - extracted annual census and absence files with
#   standardised year suffixes
# - copied loose 2019–2020 files
#
# These files are then used as inputs for
# 01_data_ingestion.ipynb.
# ==========================================================

Starting extraction from:
  C:\Users\kiero\Documents\msc-dissertation\data\raw
To:
  C:\Users\kiero\Documents\msc-dissertation\data\processed_data

--------------------------------------------------
 Extracted: england_spine_1617.csv
 Extracted: england_ks4final_1617.csv
 Extracted: england_abs_1617.csv
 Extracted: england_census_1617.csv
 Extracted: england_swf_1617.csv
 Extracted: england_spine_1718.csv
 Extracted: england_ks4final_1718.csv
 Extracted: england_abs_1718.csv
 Extracted: england_census_1718.csv
 Extracted: england_swf_1718.csv
 Extracted: england_school_information_1819.csv
 Extracted: england_ks4final_1819.csv
 Extracted: england_abs_1819.csv
 Extracted: england_census_1819.csv
 Extracted: england_swf_1819.csv
 Extracted: england_school_information_2021.csv
 Extracted: england_census_2021.csv
 Extracted: england_swf_2021.csv
 Extracted: england_school_information_2122.csv
 Extracted: england_ks4final_2122.csv
 Extracted: england_abs_2122.csv
 Extracted: england_census_